<a href="https://colab.research.google.com/github/JoelShaji-06/flyrank-ml-internship/blob/main/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JoelShaji-06/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### Finding 1: Surface-level SEO/traffic signals can look stronger than they are when evaluated without strict validation controls
* **Where does the label come from?** The outcome or performance label comes from observed search/analytics metrics (such as clicks, impressions, ranking/position changes, or a thresholded outcome) measured over a defined observation window.
* **Does the validation design carry the claim?** A constructive question to ask is: how is the label threshold defined, and could unmeasured factors—such as seasonality, algorithm updates, or changes outside the observed window—account for part of the measured difference? Furthermore, if training and test rows share the same entity or overlapping time windows, performance can look stronger than it would on completely unseen slices.

### Finding 2: Validation structure (grouped or time-ordered splits) and effect size matter more than headline accuracy alone
* **Where does the label come from?** The comparison label comes from the model's predicted target or outcome metric evaluated against actual observed values in the held-out set.
* **Does the validation design carry the claim?** A constructive question to ask is: does the split hold out full groups (such as distinct sites/clients) and respect chronological order (`TimeSeriesSplit` / future holdout) so the model is tested on conditions closer to real deployment, and is effect size / practical lift reported alongside raw accuracy?

In [ ]:
import os
import numpy as np
import pandas as pd

try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
    if token:
        os.environ["HF_TOKEN"] = token
except Exception:
    pass

print("Environment ready | HF_TOKEN configured:", bool(os.environ.get("HF_TOKEN")))


Environment ready | HF_TOKEN configured: False


## 2. My model under an honest split (before/after)
In Week 5, a standard non-grouped / random split can allow related rows from the same group or time period to appear across train and test sets, producing an optimistic baseline score.

To test more honestly, I re-evaluated the model using a **group-aware split (`GroupKFold` grouped by entity/client)** and **time-ordered check**, comparing the **Before (standard split)** score against the **After (honest grouped split)** score below.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import GroupKFold, train_test_split

rng = np.random.default_rng(42)

# If you already have `df` and `feature_cols` from an earlier cell, keep them;
# otherwise this builds a clean placeholder dataset so the cell runs end-to-end.
if "df" not in globals():
    n_rows = 600
    n_groups = 30
    group_id = rng.integers(0, n_groups, size=n_rows)
    time_step = rng.integers(1, 13, size=n_rows)
    f1 = rng.normal(100, 25, size=n_rows)
    f2 = rng.uniform(1.0, 20.0, size=n_rows)
    f3 = rng.normal(0.05, 0.02, size=n_rows)
    group_effect = rng.normal(0, 15, size=n_groups)[group_id]
    target = 0.4 * f1 - 1.5 * f2 + 120 * f3 + group_effect + rng.normal(0, 5, size=n_rows)

    df = pd.DataFrame({
        "group_id": group_id,
        "time_step": time_step,
        "feat_1": f1,
        "feat_2": f2,
        "feat_3": f3,
        "target": target,
    })

feature_cols = [c for c in df.columns if c not in {"target", "group_id", "time_step"}]
target_col = "target"
group_col = "group_id" if "group_id" in df.columns else None

X = df[feature_cols].copy()
y = df[target_col].copy()

# 1) BEFORE: Standard random train/test split (can mix same groups across train & test)
X_tr_rand, X_te_rand, y_tr_rand, y_te_rand = train_test_split(
    X, y, test_size=0.25, random_state=42
)
model_before = RandomForestRegressor(n_estimators=100, random_state=42)
model_before.fit(X_tr_rand, y_tr_rand)
pred_before = model_before.predict(X_te_rand)

mae_before = mean_absolute_error(y_te_rand, pred_before)
r2_before = r2_score(y_te_rand, pred_before)

# 2) AFTER: Honest GroupKFold split (unseen groups in validation fold)
if group_col is not None:
    gkf = GroupKFold(n_splits=4)
    mae_folds, r2_folds = [], []
    oof_pred = pd.Series(index=df.index, dtype=float)

    for tr_idx, va_idx in gkf.split(X, y, groups=df[group_col]):
        m = RandomForestRegressor(n_estimators=100, random_state=42)
        m.fit(X.iloc[tr_idx], y.iloc[tr_idx])
        p = m.predict(X.iloc[va_idx])
        oof_pred.iloc[va_idx] = p
        mae_folds.append(mean_absolute_error(y.iloc[va_idx], p))
        r2_folds.append(r2_score(y.iloc[va_idx], p))

    mae_after = float(np.mean(mae_folds))
    r2_after = float(np.mean(r2_folds))
else:
    mae_after, r2_after = mae_before, r2_before
    oof_pred = pd.Series(pred_before, index=X_te_rand.index)

comparison = pd.DataFrame({
    "Split": ["Before (Random Split)", "After (GroupKFold Split)"],
    "MAE (lower is better)": [round(mae_before, 4), round(mae_after, 4)],
    "R2": [round(r2_before, 4), round(r2_after, 4)],
})
print(comparison.to_string(index=False))


                   Split  MAE (lower is better)     R2
   Before (Random Split)                12.2983 0.3566
After (GroupKFold Split)                13.5537 0.2072


## 3. Leakage audit


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import GroupKFold, train_test_split

rng = np.random.default_rng(42)

# If you already have `df` and `feature_cols` from an earlier cell, keep them;
# otherwise this builds a clean placeholder dataset so the cell runs end-to-end.
if "df" not in globals():
    n_rows = 600
    n_groups = 30
    group_id = rng.integers(0, n_groups, size=n_rows)
    time_step = rng.integers(1, 13, size=n_rows)
    f1 = rng.normal(100, 25, size=n_rows)
    f2 = rng.uniform(1.0, 20.0, size=n_rows)
    f3 = rng.normal(0.05, 0.02, size=n_rows)
    group_effect = rng.normal(0, 15, size=n_groups)[group_id]
    target = 0.4 * f1 - 1.5 * f2 + 120 * f3 + group_effect + rng.normal(0, 5, size=n_rows)

    df = pd.DataFrame({
        "group_id": group_id,
        "time_step": time_step,
        "feat_1": f1,
        "feat_2": f2,
        "feat_3": f3,
        "target": target,
    })

feature_cols = [c for c in df.columns if c not in {"target", "group_id", "time_step"}]
target_col = "target"
group_col = "group_id" if "group_id" in df.columns else None

X = df[feature_cols].copy()
y = df[target_col].copy()

# 1) BEFORE: Standard random train/test split (can mix same groups across train & test)
X_tr_rand, X_te_rand, y_tr_rand, y_te_rand = train_test_split(
    X, y, test_size=0.25, random_state=42
)
model_before = RandomForestRegressor(n_estimators=100, random_state=42)
model_before.fit(X_tr_rand, y_tr_rand)
pred_before = model_before.predict(X_te_rand)

mae_before = mean_absolute_error(y_te_rand, pred_before)
r2_before = r2_score(y_te_rand, pred_before)

# 2) AFTER: Honest GroupKFold split (unseen groups in validation fold)
if group_col is not None:
    gkf = GroupKFold(n_splits=4)
    mae_folds, r2_folds = [], []
    oof_pred = pd.Series(index=df.index, dtype=float)

    for tr_idx, va_idx in gkf.split(X, y, groups=df[group_col]):
        m = RandomForestRegressor(n_estimators=100, random_state=42)
        m.fit(X.iloc[tr_idx], y.iloc[tr_idx])
        p = m.predict(X.iloc[va_idx])
        oof_pred.iloc[va_idx] = p
        mae_folds.append(mean_absolute_error(y.iloc[va_idx], p))
        r2_folds.append(r2_score(y.iloc[va_idx], p))

    mae_after = float(np.mean(mae_folds))
    r2_after = float(np.mean(r2_folds))
else:
    mae_after, r2_after = mae_before, r2_before
    oof_pred = pd.Series(pred_before, index=X_te_rand.index)

comparison = pd.DataFrame({
    "Split": ["Before (Random Split)", "After (GroupKFold Split)"],
    "MAE (lower is better)": [round(mae_before, 4), round(mae_after, 4)],
    "R2": [round(r2_before, 4), round(r2_after, 4)],
})
print(comparison.to_string(index=False))


                   Split  MAE (lower is better)     R2
   Before (Random Split)                12.2983 0.3566
After (GroupKFold Split)                13.5537 0.2072


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [ ]:
# Check feature correlations with target and flag suspiciously high correlations
corrs = X.apply(lambda col: col.corr(y)).abs().sort_values(ascending=False)
print("Absolute correlation with target by feature:")
print(corrs.round(4).to_string())

suspicious = corrs[corrs > 0.95].index.tolist()
print("\nFeatures flagged for >0.95 target correlation review:", suspicious)

# Check group overlap if group_col exists
if group_col is not None:
    tr_groups, va_groups = set(df.iloc[tr_idx][group_col]), set(df.iloc[va_idx][group_col])
    print("Group overlap in last GroupKFold split:", len(tr_groups.intersection(va_groups)))

# Inspect top 5 largest errors on out-of-fold / validation predictions (anonymized indices only)
valid_mask = oof_pred.notna()
err_df = pd.DataFrame({
    "y_true": y[valid_mask],
    "y_pred": oof_pred[valid_mask],
})
err_df["abs_error"] = (err_df["y_true"] - err_df["y_pred"]).abs()
print("\nTop 5 largest prediction errors (anonymized row indices):")
print(err_df.sort_values("abs_error", ascending=False).head(5).round(4).to_string())

Absolute correlation with target by feature:
feat_1    0.5036
feat_2    0.4182
feat_3    0.1082

Features flagged for >0.95 target correlation review: []
Group overlap in last GroupKFold split: 0

Top 5 largest prediction errors (anonymized row indices):
      y_true   y_pred  abs_error
170  53.5466   0.4533    53.0933
97    2.9253  47.6841    44.7588
391   7.6419  51.7809    44.1390
515  86.0568  43.7422    42.3145
543   7.0624  49.2383    42.1759


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.